# 06 - Model Evaluation (Forecast Engine)

## Objective
Evaluate the trained ML pipeline used by StockMetrics to predict **next-day returns**.
This notebook verifies whether the model meets the **ML Business Case** success criteria
and produces evaluation artefacts for the Streamlit dashboard and project documentation.

## Inputs
- Feature dataset: `data/processed/<version>/features_<version>_latest.csv`
- Trained model pipeline: `models/stock_forecast_model_<version>.pkl`

## Outputs
Saved to `outputs/<version>/`:
- Metrics report JSON: `reports/model_evaluation_report_<version>.json`
- Predictions CSVs: `reports/predictions_train_<version>.csv`, `reports/predictions_test_<version>.csv`
- Plots:
  - `figures/eval_actual_vs_pred_train_<version>.png`
  - `figures/eval_actual_vs_pred_test_<version>.png`
  - `figures/eval_residuals_hist_test_<version>.png`
  - `figures/eval_residuals_timeseries_test_<version>.png`
  - `figures/eval_pred_timeseries_test_<version>.png`
- Feature importance CSV: `reports/feature_importance_<version>.csv`

## CRISP-DM Stage
Evaluation

In [ ]:
# Make the project root importable (so `import src...` works in notebooks)
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()  # notebooks live in jupyter_notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:", PROJECT_ROOT)

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DEFAULT_VERSION, get_paths
from src.features import load_features_latest
from src.modelling import FEATURE_COLS_NUM, FEATURE_COLS_CAT, TARGET_COL, time_split
from src.evaluation import regression_metrics, save_actual_vs_pred_plot

np.random.seed(42)

In [ ]:
VERSION = DEFAULT_VERSION
paths = get_paths(VERSION)

MODELS_DIR = paths.models_dir
REPORTS_DIR = paths.outputs_dir / "reports"
FIG_DIR = paths.outputs_dir / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Version:", VERSION)
print("Models dir:", MODELS_DIR)
print("Reports dir:", REPORTS_DIR)
print("Figures dir:", FIG_DIR)

In [ ]:
feat_df = load_features_latest(paths.processed_dir, VERSION)

print("Loaded features shape:", feat_df.shape)
print("Tickers:", sorted(feat_df["Ticker"].unique().tolist()))
print("Date range:", feat_df["Date"].min().date(), "to", feat_df["Date"].max().date())

required_cols = set(FEATURE_COLS_NUM + FEATURE_COLS_CAT + [TARGET_COL, "Date"])
missing = required_cols - set(feat_df.columns)
if missing:
    raise KeyError(f"Missing required columns in features dataset: {missing}")

model_path = MODELS_DIR / f"stock_forecast_model_{VERSION}.pkl"
if not model_path.exists():
    raise FileNotFoundError(
        f"Model not found at {model_path}. Run 05_model_training.ipynb first."
    )

model = joblib.load(model_path)
print("Loaded model:", model_path.name)

In [ ]:
df = feat_df.dropna(subset=FEATURE_COLS_NUM + FEATURE_COLS_CAT + [TARGET_COL]).copy()
df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

train_df, test_df = time_split(df, test_size=0.2)

X_train = train_df[FEATURE_COLS_NUM + FEATURE_COLS_CAT]
y_train = train_df[TARGET_COL].astype(float)

X_test = test_df[FEATURE_COLS_NUM + FEATURE_COLS_CAT]
y_test = test_df[TARGET_COL].astype(float)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train date range:", train_df["Date"].min().date(), "to", train_df["Date"].max().date())
print("Test date range:", test_df["Date"].min().date(), "to", test_df["Date"].max().date())

In [ ]:
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

metrics_train = regression_metrics(y_train, pred_train)
metrics_test = regression_metrics(y_test, pred_test)

print("Train metrics:", metrics_train)
print("Test metrics:", metrics_test)

In [ ]:
train_pred_df = pd.DataFrame({
    "Date": train_df["Date"].values,
    "Ticker": train_df["Ticker"].values,
    "y_true": y_train.values,
    "y_pred": pred_train,
    "residual": (y_train.values - pred_train),
})

test_pred_df = pd.DataFrame({
    "Date": test_df["Date"].values,
    "Ticker": test_df["Ticker"].values,
    "y_true": y_test.values,
    "y_pred": pred_test,
    "residual": (y_test.values - pred_test),
})

train_csv = REPORTS_DIR / f"predictions_train_{VERSION}.csv"
test_csv = REPORTS_DIR / f"predictions_test_{VERSION}.csv"

train_pred_df.to_csv(train_csv, index=False)
test_pred_df.to_csv(test_csv, index=False)

print("Saved:", train_csv.name)
print("Saved:", test_csv.name)

In [ ]:
p1 = save_actual_vs_pred_plot(
    y_true=y_train,
    y_pred=pred_train,
    out_dir=FIG_DIR,
    filename=f"eval_actual_vs_pred_train_{VERSION}.png",
)

p2 = save_actual_vs_pred_plot(
    y_true=y_test,
    y_pred=pred_test,
    out_dir=FIG_DIR,
    filename=f"eval_actual_vs_pred_test_{VERSION}.png",
)

print("Saved:", p1.name)
print("Saved:", p2.name)